# Contrastive Learning and Adaptive Weighted Fusion for Multi-View Lung Cancer Classification Using CT Scans

This Google Colab notebook provides an educational PyTorch baseline for classifying three corresponding CT views (axial, coronal, sagittal) into `normal`, `benign`, and `malignant`.

**Before running:** prepare `train.csv`, `val.csv`, and `test.csv` with columns:
`patient_id, axial, coronal, sagittal, label`.

Each image path should be relative to the CSV's parent folder. Keep all images from a patient in only one split. This is a research prototype, not a clinical diagnostic tool.


## 1. Install dependencies

In [ ]:
!pip -q install torch torchvision pandas numpy pillow scikit-learn matplotlib

## 2. Upload dataset files

Recommended folder layout before uploading as a ZIP:

```text
data/
  train.csv
  val.csv
  test.csv
  images/
    axial/
    coronal/
    sagittal/
```

The CSV paths must match your uploaded folder layout. Upload `data.zip` below and extract it into `/content`.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os

uploaded = files.upload()  # Select data.zip
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)

if zip_name is None:
    raise ValueError("Please upload a ZIP file containing train.csv, val.csv, test.csv and images/.")

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

DATA_DIR = Path("/content/data")
print("Data directory:", DATA_DIR)
print("Files found:", [str(p) for p in DATA_DIR.glob("*")][:20])


## 3. Imports and configuration

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

from pathlib import Path

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
FEATURE_DIM = 256
PROJECTION_DIM = 128
TEMPERATURE = 0.1
CONTRASTIVE_WEIGHT = 0.2
NUM_WORKERS = 2

CLASSES = ["normal", "benign", "malignant"]
TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"
CHECKPOINT_PATH = Path("/content/best_lung_model.pth")
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("For faster training, select Runtime > Change runtime type > T4 GPU if available.")


## 4. Image preprocessing and dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAffine(degrees=5, translate=(0.02, 0.02)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

class LungCTDataset(Dataset):
    REQUIRED_COLUMNS = ["patient_id", "axial", "coronal", "sagittal", "label"]

    def __init__(self, csv_file, transform=None):
        self.csv_file = Path(csv_file)
        self.base_dir = self.csv_file.parent
        self.transform = transform
        self.data = pd.read_csv(self.csv_file)

        missing = set(self.REQUIRED_COLUMNS) - set(self.data.columns)
        if missing:
            raise ValueError(f"Missing columns in {csv_file}: {sorted(missing)}")
        if self.data[self.REQUIRED_COLUMNS].isnull().any().any():
            raise ValueError(f"Missing values found in {csv_file}")

        self.data["label"] = self.data["label"].astype(str).str.strip().str.lower()
        unknown = set(self.data["label"]) - set(CLASSES)
        if unknown:
            raise ValueError(f"Unknown labels: {unknown}. Expected labels: {CLASSES}")
        if self.data.empty:
            raise ValueError(f"No samples found in {csv_file}")

    def __len__(self):
        return len(self.data)

    def load_image(self, relative_path):
        path = self.base_dir / str(relative_path)
        if not path.is_file():
            raise FileNotFoundError(f"Image not found: {path}")
        with Image.open(path) as image:
            image = image.convert("RGB")
        return self.transform(image) if self.transform else image

    def __getitem__(self, index):
        row = self.data.iloc[index]
        views = torch.stack([
            self.load_image(row["axial"]),
            self.load_image(row["coronal"]),
            self.load_image(row["sagittal"])
        ], dim=0)
        label = CLASSES.index(row["label"])
        return views, torch.tensor(label, dtype=torch.long)

for csv_path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not csv_path.exists():
        print("Missing:", csv_path)


## 5. CNN feature extractor and adaptive fusion model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class CTFeatureExtractor(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, feature_dim),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.encoder(x)

class MultiViewContrastiveLoss(nn.Module):
    def __init__(self, temperature=TEMPERATURE):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings):
        # embeddings shape: [batch, views, projection_dim]
        batch_size, num_views, dim = embeddings.shape
        if batch_size < 2:
            return embeddings.sum() * 0.0

        embeddings = F.normalize(embeddings, dim=-1)
        embeddings = embeddings.reshape(batch_size * num_views, dim)
        similarity = embeddings @ embeddings.T / self.temperature

        sample_ids = torch.arange(
            batch_size, device=embeddings.device
        ).repeat_interleave(num_views)
        positive_mask = sample_ids[:, None].eq(sample_ids[None, :])
        self_mask = torch.eye(
            batch_size * num_views, device=embeddings.device, dtype=torch.bool
        )
        positive_mask = positive_mask & ~self_mask
        similarity = similarity.masked_fill(self_mask, float("-inf"))

        log_prob = similarity - torch.logsumexp(similarity, dim=1, keepdim=True)
        positive_count = positive_mask.sum(dim=1)
        valid = positive_count > 0
        positive_log_prob = log_prob.masked_fill(~positive_mask, 0.0).sum(dim=1)
        return -(positive_log_prob[valid] / positive_count[valid].float()).mean()

class MultiViewLungCancerModel(nn.Module):
    def __init__(self, num_classes=len(CLASSES), feature_dim=FEATURE_DIM,
                 projection_dim=PROJECTION_DIM):
        super().__init__()
        self.backbone = CTFeatureExtractor(feature_dim)
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, projection_dim),
        )
        self.view_attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, views):
        # Input [B, V, C, H, W]
        batch_size, num_views, channels, height, width = views.shape
        x = views.reshape(batch_size * num_views, channels, height, width)
        features = self.backbone(x).reshape(batch_size, num_views, -1)
        projected = self.projector(features)

        scores = self.view_attention(features).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        fused = torch.sum(features * weights.unsqueeze(-1), dim=1)
        logits = self.classifier(fused)
        return logits, projected, weights


## 6. Training and validation

In [ ]:
def run_epoch(model, loader, optimizer, classification_loss,
              contrastive_loss, device, training=True):
    model.train(training)
    total_loss = total_cls = total_con = 0.0
    all_labels, all_predictions = [], []

    for views, labels in loader:
        views, labels = views.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits, embeddings, _ = model(views)
            loss_cls = classification_loss(logits, labels)
            loss_con = contrastive_loss(embeddings)
            loss = loss_cls + CONTRASTIVE_WEIGHT * loss_con

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        n = labels.size(0)
        total_loss += loss.item() * n
        total_cls += loss_cls.item() * n
        total_con += loss_con.item() * n
        all_labels.extend(labels.detach().cpu().tolist())
        all_predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())

    size = len(loader.dataset)
    return {
        "loss": total_loss / size,
        "classification_loss": total_cls / size,
        "contrastive_loss": total_con / size,
        "accuracy": accuracy_score(all_labels, all_predictions),
    }, all_labels, all_predictions

def plot_history(history):
    df = pd.DataFrame(history)
    plt.figure(figsize=(8, 4))
    plt.plot(df.epoch, df.train_loss, label="Train loss")
    plt.plot(df.epoch, df.val_loss, label="Validation loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(df.epoch, df.train_accuracy, label="Train accuracy")
    plt.plot(df.epoch, df.val_accuracy, label="Validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True)
    plt.show()

def train_model():
    train_ds = LungCTDataset(TRAIN_CSV, train_transform)
    val_ds = LungCTDataset(VAL_CSV, eval_transform)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )

    model = MultiViewLungCancerModel().to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )
    cls_loss = nn.CrossEntropyLoss()
    con_loss = MultiViewContrastiveLoss()

    best_val_loss = float("inf")
    patience, stale_epochs = 7, 0
    history = []

    for epoch in range(EPOCHS):
        train_metrics, _, _ = run_epoch(
            model, train_loader, optimizer, cls_loss, con_loss, device, True
        )
        val_metrics, _, _ = run_epoch(
            model, val_loader, optimizer, cls_loss, con_loss, device, False
        )
        scheduler.step(val_metrics["loss"])

        row = {
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
        }
        history.append(row)
        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"train loss {row['train_loss']:.4f}, "
            f"train acc {row['train_accuracy']:.4f} | "
            f"val loss {row['val_loss']:.4f}, "
            f"val acc {row['val_accuracy']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            stale_epochs = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "classes": CLASSES,
                "image_size": IMAGE_SIZE,
                "feature_dim": FEATURE_DIM,
                "projection_dim": PROJECTION_DIM,
                "best_val_loss": best_val_loss,
            }, CHECKPOINT_PATH)
            print("Saved best checkpoint:", CHECKPOINT_PATH)
        else:
            stale_epochs += 1

        if stale_epochs >= patience:
            print("Early stopping.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)
    plot_history(history)
    return history_df

# Run training:
history_df = train_model()


## 7. Evaluate the saved model on the test set

In [ ]:
def evaluate_model():
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError("Train the model first; checkpoint not found.")

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location=device, weights_only=True
    )
    classes = checkpoint["classes"]

    model = MultiViewLungCancerModel(
        num_classes=len(classes),
        feature_dim=checkpoint["feature_dim"],
        projection_dim=checkpoint["projection_dim"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_ds = LungCTDataset(TEST_CSV, eval_transform)
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS
    )

    labels_all, preds_all, weights_all = [], [], []
    with torch.no_grad():
        for views, labels in test_loader:
            views = views.to(device)
            logits, _, weights = model(views)
            preds = logits.argmax(dim=1)
            labels_all.extend(labels.tolist())
            preds_all.extend(preds.cpu().tolist())
            weights_all.extend(weights.cpu().tolist())

    acc = accuracy_score(labels_all, preds_all)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, preds_all, labels=list(range(len(classes))),
        average="macro", zero_division=0
    )

    print(f"Test accuracy:       {acc:.4f}")
    print(f"Macro precision:     {precision:.4f}")
    print(f"Macro recall:        {recall:.4f}")
    print(f"Macro F1-score:      {f1:.4f}")
    print("\nClassification report:")
    report = classification_report(
        labels_all, preds_all, labels=list(range(len(classes))),
        target_names=classes, zero_division=0
    )
    print(report)

    matrix = confusion_matrix(
        labels_all, preds_all, labels=list(range(len(classes)))
    )
    print("Confusion matrix:\n", matrix)

    (RESULTS_DIR / "classification_report.txt").write_text(
        report, encoding="utf-8"
    )
    pd.DataFrame(matrix, index=classes, columns=classes).to_csv(
        RESULTS_DIR / "confusion_matrix.csv"
    )

    plt.figure(figsize=(6, 5))
    plt.imshow(matrix, interpolation="nearest")
    plt.title("Test Confusion Matrix")
    plt.colorbar()
    positions = np.arange(len(classes))
    plt.xticks(positions, classes, rotation=30)
    plt.yticks(positions, classes)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, str(matrix[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.show()

    avg_weights = np.asarray(weights_all).mean(axis=0)
    view_names = ["Axial", "Coronal", "Sagittal"]
    print("\nMean adaptive view weights:")
    for name, weight in zip(view_names, avg_weights):
        print(f"{name}: {weight:.4f}")

    pd.DataFrame({
        "view": view_names, "average_weight": avg_weights
    }).to_csv(RESULTS_DIR / "view_weights.csv", index=False)

evaluate_model()


## 8. Predict one three-view sample

Change the three paths to images from the same CT examination. The returned scores are model outputs and are **not calibrated probabilities or a medical diagnosis**.


In [ ]:
def predict_one(axial_path, coronal_path, sagittal_path):
    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location=device, weights_only=True
    )
    classes = checkpoint["classes"]
    model = MultiViewLungCancerModel(
        num_classes=len(classes),
        feature_dim=checkpoint["feature_dim"],
        projection_dim=checkpoint["projection_dim"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    images = []
    for path in [axial_path, coronal_path, sagittal_path]:
        with Image.open(path) as image:
            images.append(eval_transform(image.convert("RGB")))

    views = torch.stack(images).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _, weights = model(views)
        scores = F.softmax(logits, dim=1)[0]
        pred_idx = int(scores.argmax().item())

    print("Experimental predicted class:", classes[pred_idx])
    print("Model scores:")
    for name, score in zip(classes, scores.cpu().tolist()):
        print(f"  {name}: {score:.4f}")
    print("Adaptive view weights:")
    for name, weight in zip(
        ["Axial", "Coronal", "Sagittal"], weights[0].cpu().tolist()
    ):
        print(f"  {name}: {weight:.4f}")

# Example (uncomment and edit):
# predict_one(
#     "/content/data/images/axial/P001.png",
#     "/content/data/images/coronal/P001.png",
#     "/content/data/images/sagittal/P001.png",
# )


## 9. Download trained model and results

In [ ]:
from google.colab import files
import shutil

# Download the checkpoint:
files.download(str(CHECKPOINT_PATH))

# Optionally package evaluation results into a ZIP:
results_zip = shutil.make_archive(
    "/content/lung_cancer_results", "zip", root_dir=str(RESULTS_DIR)
)
files.download(results_zip)
